# 第 01 章 数据导入与对象检查

## 学习目标

读取本项目登记的样本并保留来源，理解细胞、基因和计数矩阵。

## 为什么做与怎样做

按 project.json 的格式读取 H5、MTX 或已确认 counts 的 H5AD；按共有基因合并。

前置章节：00。运行前请完成项目环境准备。

本章在项目副本运行。遇到等待材料/确认，请按根目录 **99_运行与AI协作指南.md** 查看当前结果、与用户讨论并续跑；不能跳过待办直接进入下游。


In [ ]:
# 功能说明：查找公共环境和当前项目，读取有效上游检查点；模板副本之间不共享分析状态。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
from __future__ import annotations
from pathlib import Path
import os
import sys
ROOT = Path(os.environ.get("SC_COURSE_ROOT", Path.cwd())).resolve()
while not (ROOT / "config/course.json").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "config/course.json").is_file():
    raise RuntimeError("请从课程目录或其 notebooks/exercises 目录运行。")
os.environ["CELLTYPIST_FOLDER"] = str(ROOT / ".runtime/celltypist")
os.environ["MPLCONFIGDIR"] = str(ROOT / ".runtime/matplotlib")
sys.path.insert(0, str(ROOT / "tools"))
from course_runtime import start_chapter, marker_sets, scaled_view
from course_projects import resolve_project, input_path, read_sample
from course_resources import qc_gene_sets, marker_resources, resolution_preview
import anndata as ad
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from scipy.sparse import csr_matrix
from scipy.stats import median_abs_deviation
INSTALL_ROOT = ROOT
ROOT = resolve_project()
ctx = start_chapter("01")



## 01.1 数据导入

本课程使用 NeurIPS 2021 单细胞多组学基准数据集中的两个 10X H5 文件：s1d1 与 s1d3。每个文件包含 36,601 个 Gene Expression 特征和 140 个 Antibody Capture 特征；read_10x_h5 默认只读取 Gene Expression。本课程分析 RNA 计数。样本标识保留为 samples，不把样本差异自动解释为纯技术效应。

来源：[数据集页面](https://figshare.com/articles/dataset/NeurIPS_2021_Benchmark_dataset/22716739)、[Scanpy 教程](https://scanpy.scverse.org/en/stable/tutorials/basics/clustering.html)。

课程包已包含所需 H5 数据。需要核对来源时，可访问上面的数据集页面。

In [ ]:
# 功能说明：读取本项目登记样本的原始计数矩阵（教程为两个 10x HDF5 样本），合并为单一 AnnData。
# 运行目的：构建包含所有细胞的综合对象并保留样本来源标签，用于后续 QC 与分析。
# 变量/函数/参数解析（逐项）：
# - samples(dict)：样本 ID 到材料说明的映射；含 path、format、capture_library 和原始计数确认。教程为 s1d1、s1d3。
# - adatas(dict)：用于暂存每个样本的 AnnData。
# - for sample_id, filename in samples.items()：遍历样本映射。
#   - ROOT / "data" / "h5" / filename：定位课程内的本地 H5 文件。
#   - sc.read_10x_h5(path)：读取 10x 格式 HDF5 计数矩阵为 AnnData。
#   - sample_adata.var_names_make_unique()：唯一化基因名，避免重复导致冲突。
#   - adatas[sample_id] = sample_adata：以样本 ID 为键保存。
# - ad.concat(adatas, label="samples")：按细胞维度拼接多个 AnnData，并在 `obs['samples']` 写入来源标签。join='inner'  ['inner', 'outer'] (默认值: 'inner')指定拼接时数值的对齐方式。若选择“outer”，则取其他轴的并集；若选择“inner”，则取交集。
# - adata.obs_names_make_unique()：唯一化细胞名，避免重复。
# - print(adata.obs["samples"].value_counts())：打印各样本细胞数量统计。
# 数据流程：
# - 输入：两个 10x .h5 计数文件。
# - 输出：合并后的 `adata`（细胞来自两个样本，基因为共有基因（交集））。

samples = ctx.config["samples"]
adatas = {}

for sample_id, sample_spec in samples.items():
    sample_adata = read_sample(ROOT, sample_spec)
    sample_adata.obs["capture_library"] = sample_spec.get("capture_library", sample_id)
    sample_adata.var_names_make_unique()
    adatas[sample_id] = sample_adata

adata = ad.concat(adatas, label="samples",join='inner', merge='same')
adata.obs_names_make_unique()
print(adata.obs["samples"].value_counts())


读取数据后，scanpy 会显示警告，指出并非所有变量名称都是唯一的。 这表明某些变量（此处为基因）出现不止一次，这可能会导致下游分析任务出现错误或意外行为。 我们执行建议的函数 var_names_make_unique()，它通过向每个重复的索引元素附加一个数字字符串使变量名称唯一：‘1’，‘2’ 等。

In [ ]:
# 功能说明：查看 AnnData 对象的摘要信息。
# 运行目的：检查数据加载是否成功，查看细胞数（n_obs）、基因数（n_vars）及已有的注释信息。
# 详细代码解析：
# 1. `adata`
#    - 在 Jupyter Notebook 中直接输入变量名，会调用其 `__repr__` 方法，打印对象的概览。
#    - 输出通常包含：
#      - `n_obs × n_vars`: 细胞数 × 基因数。
#      - `obs`: 细胞的观测注释（如样本来源）。
#      - `var`: 基因的特征注释（如基因名）。
adata

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
print(adata.X)


In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
adata.obs


每个样本约有 8,000 个条形码；实际细胞数和基因数以本次读取结果为准。

## 保存本章结果

保存表格、参数摘要和可供后续章节读取的数据。

In [ ]:
# 功能说明：运行本章的检查、绘图或结果保存；输入对象及输出目录均来自当前项目。
# 运行目的：让当前项目的输入、候选与用户决定对应，避免复用其他分析的答案。
# 数据流程：ctx 定位本项目；中间证据进入 results；确认后才形成下游检查点。
ctx.table("sample_counts", adata.obs["samples"].value_counts().rename("n_cells"))
ctx.finish(adata, {"samples": adata.obs["samples"].value_counts().to_dict()})


## 结果阅读与思考

请打开本次结果目录中的 summary.json、tables 和 figures。将目的、方法、结果和解释写入本章报告源稿，再更新 Word。

思考题：样本合并以后，为什么细胞索引仍然需要唯一？

运行与报告操作见课程根目录的 99_运行与AI协作指南.md。